<a href="https://colab.research.google.com/github/chadallen/insider_trading_detection/blob/main/Detect_insider_trading_on_prediction_markets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook looks for suspicious trading patterns in resolved Polymarket prediction markets.

**How it works**
1. We pull ~120 recently resolved markets and score each one across four signals:

2. VPIN — Models for detecting insider trading in
*financial markets* (Easley, López de Prado & O'Hara, 2011, 2012)
3. Time-weighted VPIN — same, but weighted toward moves closer to resolution
4. Resolution surprise — how wrong was the market's implied probability vs. what actually happened?
5. Late move ratio — what % of total price movement happened right before resolution?
6. We then run Isolation Forest across all four signals to rank markets by how anomalous they look.

Note: Polymarket only gives us 12h price granularity for closed markets, so VPIN is approximated from price movement rather than actual order flow.
No labeled training data yet — this is unsupervised
~120 markets is a small sample



In [56]:
#@title 1 - Install dependencies

!pip install requests pandas numpy scikit-learn plotly -q

In [57]:
#@title 2 - Fetch resolved markets

import requests
import pandas as pd
import json
from datetime import datetime, timedelta, timezone

def fetch_resolved_markets(days_back=365, limit=200):
    """Fetch recently resolved markets with token IDs included."""

    end_date = datetime.now(timezone.utc)
    start_date = end_date - timedelta(days=days_back)

    url = "https://gamma-api.polymarket.com/markets"
    params = {
        "closed": "true",
        "limit": limit,
        "after": start_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "order": "endDate",
        "ascending": "false",
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    markets = []
    for m in data:
        # parse token IDs
        raw_tokens = m.get("clobTokenIds", "[]")
        try:
            tokens = json.loads(raw_tokens) if isinstance(raw_tokens, str) else raw_tokens
            token_id = tokens[0] if tokens else None
        except:
            token_id = None

        markets.append({
            "market_id":       m.get("conditionId"),
            "token_id":        token_id,
            "question":        m.get("question"),
            "resolution_time": m.get("endDate"),
            "final_price":     m.get("outcomePrices"),
            "volume":          float(m.get("volume") or 0),
        })

    df = pd.DataFrame(markets)
    # drop rows with no token_id or no volume
    df = df[df["token_id"].notna() & (df["volume"] > 0)]
    print(f"Fetched {len(df)} markets with trade data")
    return df

df_markets = fetch_resolved_markets(days_back=548)
print(df_markets[["question", "volume", "token_id"]].head(5))

Fetched 182 markets with trade data
                                        question    volume  \
0     idOS FDV above $100M one day after launch?  9606.287   
1      idOS FDV above $50M one day after launch? 21937.653   
2  Espresso FDV above $50M one day after launch? 83280.458   
3   Espresso FDV above $1B one day after launch? 79468.370   
4      idOS FDV above $80M one day after launch? 12133.768   

                                                      token_id  
0  66871872422197194573137055386260077430224454921744173428...  
1  73268068968476052541700461224016254385370856675658988668...  
2  96267458254090441798245236969531476202789337333051349091...  
3  38668657318576350037621534664851279396445838979397040424...  
4  75106699590274454429918698609373837985540110268405910990...  


In [58]:
#@title 3 - Fetch price histories

def fetch_price_history(token_id, resolution_time, hours_before=48):
    """Fetch price history - note: closed markets only support 12h+ granularity."""

    if isinstance(resolution_time, str):
        res_time = datetime.fromisoformat(resolution_time.replace("Z", "+00:00"))
    else:
        res_time = resolution_time

    start_time = res_time - timedelta(hours=hours_before)

    # correct host: clob not gamma-api
    url = "https://clob.polymarket.com/prices-history"
    params = {
        "market":    token_id,
        "interval":  "max",
        "fidelity":  720,   # 12 hours in minutes - minimum for closed markets
        "startTs":   int(start_time.timestamp()),
        "endTs":     int(res_time.timestamp()),
    }

    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()

        if not data or "history" not in data or not data["history"]:
            return pd.DataFrame()

        # columns are 't' and 'p' not 'timestamp' and 'price'
        df = pd.DataFrame(data["history"])
        df = df.rename(columns={"t": "timestamp", "p": "price"})
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
        df["price"] = df["price"].astype(float)
        return df

    except Exception as e:
        print(f"  Error: {e}")
        return pd.DataFrame()

# Test on first 5 markets
for _, row in df_markets.head(5).iterrows():
    history = fetch_price_history(row["token_id"], row["resolution_time"])
    print(f"Market: {row['question'][:55]}...")
    print(f"  Points: {len(history)}  |  Volume: ${row['volume']:,.0f}")
    if len(history) > 0:
        print(f"  Price: {history['price'].iloc[0]:.2f} → {history['price'].iloc[-1]:.2f}")
    print()

Market: idOS FDV above $100M one day after launch?...
  Points: 6  |  Volume: $9,606
  Price: 0.39 → 0.00

Market: idOS FDV above $50M one day after launch?...
  Points: 6  |  Volume: $21,938
  Price: 0.46 → 0.02

Market: Espresso FDV above $50M one day after launch?...
  Points: 8  |  Volume: $83,280
  Price: 0.99 → 1.00

Market: Espresso FDV above $1B one day after launch?...
  Points: 7  |  Volume: $79,468
  Price: 0.02 → 0.00

Market: idOS FDV above $80M one day after launch?...
  Points: 6  |  Volume: $12,134
  Price: 0.41 → 0.01



In [59]:
#@title 4 - Compute VPIN and price features

import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

def compute_vpin_from_prices(history_df):
    if len(history_df) < 3:
        return None
    prices = history_df["price"].values
    changes = np.diff(prices)
    buy_pressure = np.where(changes > 0, changes, 0).sum()
    sell_pressure = np.where(changes < 0, -changes, 0).sum()
    total = buy_pressure + sell_pressure
    if total == 0:
        return 0.0
    return abs(buy_pressure - sell_pressure) / total

def compute_price_features(history_df):
    if len(history_df) < 2:
        return None
    prices = history_df["price"].values
    changes = np.abs(np.diff(prices))
    return {
        "price_volatility": changes.std() if len(changes) > 1 else 0,
        "max_single_move": changes.max() if len(changes) > 0 else 0,
        "final_price": prices[-1],
        "starting_price": prices[0],
        "total_price_move": abs(prices[-1] - prices[0]),
    }

histories = {}  # NEW — cache for Cell 4b

results = []
for _, row in df_markets.iterrows():
    history = fetch_price_history(row["token_id"], row["resolution_time"])
    if len(history) < 3:
        continue

    starting_price = history["price"].iloc[0]
    if not (0.15 <= starting_price <= 0.85):
        continue

    histories[row["token_id"]] = history  # NEW — store history for reuse

    vpin = compute_vpin_from_prices(history)
    features = compute_price_features(history)
    if vpin is None or features is None:
        continue

    results.append({
        "question": row["question"],
        "volume": row["volume"],
        "vpin_score": vpin,
        **features,
    })

df_scored = pd.DataFrame(results)
print(f"Markets with enough data: {len(df_scored)}")

feature_cols = ["vpin_score", "volume", "total_price_move", "price_volatility", "max_single_move"]
X = df_scored[feature_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
model = IsolationForest(contamination=0.1, random_state=42)
df_scored["anomaly_score"] = model.fit_predict(X_scaled)
df_scored["suspicion_score"] = -model.decision_function(X_scaled)

top = df_scored.nlargest(15, "suspicion_score")[
    ["question", "suspicion_score", "vpin_score", "volume", "starting_price", "final_price"]
].reset_index(drop=True)
print(top.to_string())

Markets with enough data: 132
                                               question  suspicion_score  vpin_score      volume  starting_price  final_price
0            Stable FDV above $2B one day after launch?            0.139       0.269 2108189.722           0.770        0.002
1   Over $500K committed to the Sol Raffle public sale?            0.110       0.988   45128.015           0.765        0.007
2           Owlto FDV above $200M one day after launch?            0.082       0.588   42670.776           0.205        0.992
3          Seeker FDV above $100M one day after launch?            0.081       0.013  314602.962           0.730        0.755
4             Trove FDV above $5M one day after launch?            0.073       1.000  234777.441           0.815        0.001
5           Lighter FDV above $3B one day after launch?            0.056       0.880 1203029.902           0.530        0.001
6         Kinetiq FDV above $250M one day after launch?            0.044       0.318  83

In [60]:
#@title 5 - Compute resolution surprise features

def compute_resolution_surprise(history_df):
    if len(history_df) < 2:
        return None, None
    prices = history_df["price"].values
    actual_resolution = 1.0 if prices[-1] > 0.5 else 0.0
    prior_price = prices[0]
    surprise_score = abs(actual_resolution - prior_price)
    total_move = abs(prices[-1] - prices[0])
    final_step_move = abs(prices[-1] - prices[-2])
    late_move_ratio = (final_step_move / total_move) if total_move > 0.01 else 0.0
    return surprise_score, late_move_ratio

surprise_scores = []
late_move_ratios = []

for _, row in df_scored.iterrows():
    token_id = df_markets.loc[
        df_markets["question"] == row["question"], "token_id"
    ].values[0]
    history = histories.get(token_id)
    surprise, late_move = compute_resolution_surprise(history) if history is not None else (0.0, 0.0)
    surprise_scores.append(surprise)
    late_move_ratios.append(late_move)

df_scored["surprise_score"] = surprise_scores
df_scored["late_move_ratio"] = late_move_ratios

print(df_scored[["question", "surprise_score", "late_move_ratio"]].head(10).to_string())

                                            question  surprise_score  late_move_ratio
0         idOS FDV above $100M one day after launch?           0.385            0.038
1          idOS FDV above $50M one day after launch?           0.455            0.160
2          idOS FDV above $80M one day after launch?           0.405            0.027
3     Espresso FDV above $200M one day after launch?           0.550            0.845
4     Espresso FDV above $400M one day after launch?           0.265            0.011
5         idOS FDV above $200M one day after launch?           0.230            0.105
6     Espresso FDV above $300M one day after launch?           0.480            0.014
7          idOS FDV above $20M one day after launch?           0.410            0.425
8       ETHGAS FDV above $500M one day after launch?           0.305            0.195
9  Will Hyperliquid dip to $28 by December 31, 2026?           0.365            0.104


In [61]:
#@title  6 - Compute time-weighted VPIN

def compute_time_weighted_vpin(history_df):
    """
    Like VPIN, but movements closer to resolution are weighted more heavily.
    Weight increases linearly: first move = 1, last move = N (number of steps).
    """
    if len(history_df) < 3:
        return None

    prices = history_df["price"].values
    changes = np.diff(prices)
    n = len(changes)

    # Linear weights: 1, 2, 3 ... N (last move gets highest weight)
    weights = np.arange(1, n + 1, dtype=float)

    weighted_buy = np.where(changes > 0, changes * weights, 0).sum()
    weighted_sell = np.where(changes < 0, -changes * weights, 0).sum()
    total = weighted_buy + weighted_sell

    if total == 0:
        return 0.0
    return abs(weighted_buy - weighted_sell) / total

# Add to df_scored
tw_vpins = []
for _, row in df_scored.iterrows():
    token_id = df_markets.loc[
        df_markets["question"] == row["question"], "token_id"
    ].values[0]
    history = histories.get(token_id)
    tw_vpin = compute_time_weighted_vpin(history) if history is not None else 0.0
    tw_vpins.append(tw_vpin)

df_scored["time_weighted_vpin"] = tw_vpins
print(df_scored[["question", "vpin_score", "time_weighted_vpin"]].head(10).to_string())

                                            question  vpin_score  time_weighted_vpin
0         idOS FDV above $100M one day after launch?       1.000               1.000
1          idOS FDV above $50M one day after launch?       1.000               1.000
2          idOS FDV above $80M one day after launch?       1.000               1.000
3     Espresso FDV above $200M one day after launch?       0.466               0.435
4     Espresso FDV above $400M one day after launch?       0.726               0.690
5         idOS FDV above $200M one day after launch?       0.746               0.702
6     Espresso FDV above $300M one day after launch?       0.906               0.919
7          idOS FDV above $20M one day after launch?       1.000               1.000
8       ETHGAS FDV above $500M one day after launch?       0.442               0.348
9  Will Hyperliquid dip to $28 by December 31, 2026?       0.669               0.686


In [66]:
#@title 7 - Score markets with Isolation Forest

feature_cols = [
    "vpin_score", "time_weighted_vpin",  # ← add this
    "volume", "total_price_move",
    "price_volatility", "max_single_move",
    "surprise_score", "late_move_ratio"
]
X = df_scored[feature_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
model = IsolationForest(contamination=0.1, random_state=42)
df_scored["anomaly_score"] = model.fit_predict(X_scaled)
df_scored["suspicion_score"] = -model.decision_function(X_scaled)

top = df_scored.nlargest(15, "suspicion_score")[
    ["question", "suspicion_score", "vpin_score", "surprise_score",
     "late_move_ratio", "volume", "starting_price", "final_price"]
].reset_index(drop=True)
print(top.to_string())


                                               question  suspicion_score  vpin_score  surprise_score  late_move_ratio      volume  starting_price  final_price
0          Seeker FDV above $100M one day after launch?            0.178       0.013           0.270           18.400  314602.962           0.730        0.755
1            Stable FDV above $2B one day after launch?            0.109       0.269           0.770            0.226 2108189.722           0.770        0.002
2   Over $500K committed to the Sol Raffle public sale?            0.090       0.988           0.765            0.006   45128.015           0.765        0.007
3           Owlto FDV above $200M one day after launch?            0.083       0.588           0.795            0.716   42670.776           0.205        0.992
4             Trove FDV above $5M one day after launch?            0.069       1.000           0.815            0.079  234777.441           0.815        0.001
5         Over $20M committed to the Space pub

In [64]:
#@title 8 - Export and download results

df_scored.nlargest(30, "suspicion_score").to_csv("polymarket_suspicious.csv", index=False)
from google.colab import files
#skip for now
# files.download("polymarket_suspicious.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [65]:
#@title 9 - Preview results

print(df_scored.nlargest(30, "suspicion_score").to_csv(index=False))

question,volume,vpin_score,price_volatility,max_single_move,final_price,starting_price,total_price_move,anomaly_score,suspicion_score,surprise_score,late_move_ratio,time_weighted_vpin
Seeker FDV above $100M one day after launch?,314602.962484,0.013123359580052504,0.1206119058547291,0.46,0.755,0.73,0.025000000000000022,-1,0.2059357356351944,0.27,18.399999999999984,0.006289308176100654
Stable FDV above $2B one day after launch?,2108189.721855,0.26884729753367154,0.06717622875598588,0.56,0.0015,0.77,0.7685000000000001,-1,0.1414396162946141,0.77,0.22576447625243978,0.45790710972742876
Over $500K committed to the Sol Raffle public sale?,45128.015107,0.9882583170254404,0.19638844638878325,0.497,0.0075,0.765,0.7575000000000001,-1,0.08799145519780294,0.765,0.00594059405940594,0.9800995024875622
Trove FDV above $5M one day after launch?,234777.440695,1.0,0.14214654410150107,0.3949999999999999,0.0005,0.815,0.8145,-1,0.08503069560099341,0.815,0.07918968692449356,1.0
Owlto FDV above $200M one day 

In [68]:
#@title 10 - Load labeled insider trading cases

import io

LABELS_CSV = """market_question,label
"Will the US strike Iran by February 28, 2026?",SUSPECTED
"Khamenei out as Supreme Leader of Iran by March 31, 2026?",SUSPECTED
"Will Nicolás Maduro be removed from office by January 31, 2026?",SUSPECTED
"Will the US take military action against Venezuela by January 31, 2026?",SUSPECTED
"Will María Corina Machado win the 2025 Nobel Peace Prize?",CONFIRMED
"Will Israel strike Iran between June 12–18, 2025?",CONFIRMED
"Which crypto company will ZachXBT expose for insider trading?",SUSPECTED
"Will Taylor Swift and Travis Kelce get engaged in 2025?",SUSPECTED
"Who will be the most searched person on Google in 2025?",SUSPECTED
"Will Google release Gemini 3.0 Flash on specific date window?",POSSIBLE
"Will the 2024 US presidential election be won by Donald Trump?",POSSIBLE
"""

df_labels = pd.read_csv(io.StringIO(LABELS_CSV))
print(f"Loaded {len(df_labels)} labeled cases")
print(df_labels[["market_question", "label"]].to_string())

Loaded 11 labeled cases
                                                            market_question      label
0                             Will the US strike Iran by February 28, 2026?  SUSPECTED
1                 Khamenei out as Supreme Leader of Iran by March 31, 2026?  SUSPECTED
2           Will Nicolás Maduro be removed from office by January 31, 2026?  SUSPECTED
3   Will the US take military action against Venezuela by January 31, 2026?  SUSPECTED
4                 Will María Corina Machado win the 2025 Nobel Peace Prize?  CONFIRMED
5                         Will Israel strike Iran between June 12–18, 2025?  CONFIRMED
6             Which crypto company will ZachXBT expose for insider trading?  SUSPECTED
7                   Will Taylor Swift and Travis Kelce get engaged in 2025?  SUSPECTED
8                   Who will be the most searched person on Google in 2025?  SUSPECTED
9             Will Google release Gemini 3.0 Flash on specific date window?   POSSIBLE
10           Will t

In [69]:
#@title 10a - Look up token IDs from question text

def lookup_token_id(question, verbose=True):
    url = "https://gamma-api.polymarket.com/markets"
    keywords = [
        question[:50],
        question[:30],
        question.split("?")[0][-30:],
    ]
    for kw in keywords:
        params = {"question": kw, "limit": 5}
        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            data = r.json()
            if not data:
                continue
            for m in data:
                fetched_q = m.get("question", "").lower()
                if any(word in fetched_q for word in question.lower().split()[:5]):
                    raw_tokens = m.get("clobTokenIds", "[]")
                    try:
                        tokens = json.loads(raw_tokens) if isinstance(raw_tokens, str) else raw_tokens
                        token_id = tokens[0] if tokens else None
                    except:
                        token_id = None
                    if token_id:
                        if verbose:
                            print(f"✅ FOUND:    {question[:55]}")
                            print(f"   matched:  {m.get('question', '')[:55]}")
                            print(f"   end_date: {m.get('endDate')}")
                        return token_id, m.get("endDate"), m.get("question")
        except Exception as e:
            print(f"  API error: {e}")
            continue
    if verbose:
        print(f"❌ NOT FOUND: {question[:60]}")
    return None, None, None

print("Looking up token IDs for labeled markets...\n")
lookup_results = []
for _, row in df_labels.iterrows():
    token_id, end_date, matched_q = lookup_token_id(row["market_question"])
    lookup_results.append({
        "label_question": row["market_question"],
        "label": row["label"],
        "matched_question": matched_q,
        "token_id": token_id,
        "end_date": end_date,
    })
    print()

df_lookup = pd.DataFrame(lookup_results)
found = df_lookup["token_id"].notna().sum()
print(f"── Summary ──────────────────────────────────────")
print(f"Token IDs found: {found}/{len(df_labels)}")

Looking up token IDs for labeled markets...

✅ FOUND:    Will the US strike Iran by February 28, 2026?
   matched:  Will Joe Biden get Coronavirus before the election?
   end_date: 2020-11-04T00:00:00Z

✅ FOUND:    Khamenei out as Supreme Leader of Iran by March 31, 202
   matched:  Will a new Supreme Court Justice be confirmed before No
   end_date: 2020-11-04T00:00:00Z

✅ FOUND:    Will Nicolás Maduro be removed from office by January 3
   matched:  Will Joe Biden get Coronavirus before the election?
   end_date: 2020-11-04T00:00:00Z

✅ FOUND:    Will the US take military action against Venezuela by J
   matched:  Will Joe Biden get Coronavirus before the election?
   end_date: 2020-11-04T00:00:00Z

✅ FOUND:    Will María Corina Machado win the 2025 Nobel Peace Priz
   matched:  Will Joe Biden get Coronavirus before the election?
   end_date: 2020-11-04T00:00:00Z

✅ FOUND:    Will Israel strike Iran between June 12–18, 2025?
   matched:  Will Joe Biden get Coronavirus before the elec

In [70]:
#@title 10b - Score labeled markets and compare to model

validation_results = []

for _, row in df_lookup.iterrows():
    if row["token_id"] is None:
        validation_results.append({
            "question": row["label_question"],
            "label": row["label"],
            "found": False,
            "suspicion_score": None,
            "vpin_score": None,
            "surprise_score": None,
            "late_move_ratio": None,
            "starting_price": None,
        })
        continue

    history = fetch_price_history(row["token_id"], row["end_date"])

    if len(history) < 3:
        print(f"⚠️  NO HISTORY: {row['label_question'][:55]}")
        validation_results.append({
            "question": row["label_question"],
            "label": row["label"],
            "found": True,
            "suspicion_score": None,
            "vpin_score": None,
            "surprise_score": None,
            "late_move_ratio": None,
            "starting_price": history["price"].iloc[0] if len(history) > 0 else None,
        })
        continue

    vpin = compute_vpin_from_prices(history)
    features = compute_price_features(history)
    surprise, late_move = compute_resolution_surprise(history)
    tw_vpin = compute_time_weighted_vpin(history)

    if any(v is None for v in [vpin, features, surprise, late_move, tw_vpin]):
        continue

    # Score using existing model
    feature_vals = [[
        vpin, tw_vpin, row.get("volume", 0),
        features["total_price_move"], features["price_volatility"],
        features["max_single_move"], surprise, late_move
    ]]
    suspicion = -model.decision_function(scaler.transform(feature_vals))[0]

    print(f"✅ SCORED: {row['label_question'][:55]} → {suspicion:.3f}")
    validation_results.append({
        "question": row["label_question"],
        "label": row["label"],
        "found": True,
        "suspicion_score": suspicion,
        "vpin_score": vpin,
        "surprise_score": surprise,
        "late_move_ratio": late_move,
        "starting_price": features["starting_price"],
        "final_price": features["final_price"],
    })

df_validation = pd.DataFrame(validation_results)

print("\n── Validation Results ───────────────────────────────────────────")
print(df_validation[[
    "question", "label", "found",
    "suspicion_score", "vpin_score", "surprise_score", "late_move_ratio"
]].to_string())

scored = df_validation["suspicion_score"].notna()
if scored.sum() > 0:
    print(f"\nAvg suspicion score (labeled markets): {df_validation.loc[scored, 'suspicion_score'].mean():.3f}")
    print(f"Avg suspicion score (our top 15):      {df_scored.nlargest(15, 'suspicion_score')['suspicion_score'].mean():.3f}")

⚠️  NO HISTORY: Will the US strike Iran by February 28, 2026?
⚠️  NO HISTORY: Khamenei out as Supreme Leader of Iran by March 31, 202
⚠️  NO HISTORY: Will Nicolás Maduro be removed from office by January 3
⚠️  NO HISTORY: Will the US take military action against Venezuela by J
⚠️  NO HISTORY: Will María Corina Machado win the 2025 Nobel Peace Priz
⚠️  NO HISTORY: Will Israel strike Iran between June 12–18, 2025?
⚠️  NO HISTORY: Which crypto company will ZachXBT expose for insider tr
⚠️  NO HISTORY: Will Taylor Swift and Travis Kelce get engaged in 2025?
⚠️  NO HISTORY: Who will be the most searched person on Google in 2025?
⚠️  NO HISTORY: Will Google release Gemini 3.0 Flash on specific date w
⚠️  NO HISTORY: Will the 2024 US presidential election be won by Donald

── Validation Results ───────────────────────────────────────────
                                                                   question      label  found suspicion_score vpin_score surprise_score late_move_ratio
0    

In [90]:
#@title 11 - Fetch labeled market trades from Dune API

import requests
import time
from google.colab import userdata

DUNE_API_KEY = userdata.get('DUNE_API_KEY')

HEADERS = {
    "X-Dune-Api-Key": DUNE_API_KEY,
    "Content-Type": "application/json"
}

def execute_sql(sql):
    r = requests.post(
        "https://api.dune.com/api/v1/sql/execute",
        headers=HEADERS,
        json={"sql": sql, "performance": "medium"}
    )
    r.raise_for_status()
    return r.json()["execution_id"]

def poll_until_done(execution_id, timeout=120, interval=5):
    start = time.time()
    while time.time() - start < timeout:
        r = requests.get(
            f"https://api.dune.com/api/v1/execution/{execution_id}/status",
            headers=HEADERS
        )
        r.raise_for_status()
        state = r.json()["state"]
        if state == "QUERY_STATE_COMPLETED":
            return True
        elif state in ("QUERY_STATE_FAILED", "QUERY_STATE_CANCELLED"):
            print(f"  Query failed: {state}")
            return False
        print(f"  Status: {state} — waiting {interval}s...")
        time.sleep(interval)
    print("  Timed out.")
    return False

def fetch_results(execution_id):
    r = requests.get(
        f"https://api.dune.com/api/v1/execution/{execution_id}/results",
        headers=HEADERS
    )
    r.raise_for_status()
    rows = r.json().get("result", {}).get("rows", [])
    return pd.DataFrame(rows)

# ── Step 1: Find exact question text for missing markets ─────────────────────
print("── Step 1: Finding exact question text ──────────────────────────")
diagnostic_sql = """
    SELECT DISTINCT question, COUNT(*) as trade_count
    FROM polymarket_polygon.market_trades
    WHERE (
        question LIKE '%Machado%'
        OR question LIKE '%Israel%strike%'
        OR question LIKE '%Axiom%insider%'
    )
    AND block_time >= TIMESTAMP '2025-01-01'
    GROUP BY question
    ORDER BY trade_count DESC
    LIMIT 20
"""
exec_id = execute_sql(diagnostic_sql)
poll_until_done(exec_id)
df_diag = fetch_results(exec_id)
print(df_diag[["question", "trade_count"]].to_string())

# ── Step 2: Fetch trade data for known markets ───────────────────────────────
print("\n── Step 2: Fetching trade data ──────────────────────────────────")

QUERIES = {
    "maduro": {
        "label": "SUSPECTED",
        "sql": """
            SELECT block_time, question, asset_id, price, amount, shares, maker, taker, token_outcome_name
            FROM polymarket_polygon.market_trades
            WHERE question LIKE '%Maduro%'
              AND block_time BETWEEN TIMESTAMP '2026-01-28' AND TIMESTAMP '2026-01-31'
            ORDER BY block_time ASC
        """
    },
    "zachxbt": {
        "label": "SUSPECTED",
        "sql": """
            SELECT block_time, question, asset_id, price, amount, shares, maker, taker, token_outcome_name
            FROM polymarket_polygon.market_trades
            WHERE question = 'Will Axiom be accused of insider trading?'
              AND block_time BETWEEN TIMESTAMP '2026-02-24' AND TIMESTAMP '2026-02-27'
            ORDER BY block_time ASC
        """
    },
    "nobel": {
    "label": "CONFIRMED",
    "sql": """
        SELECT block_time, question, asset_id, price, amount, shares, maker, taker, token_outcome_name
        FROM polymarket_polygon.market_trades
        WHERE question = 'Will María Corina Machado win the Nobel Peace Prize in 2025?'
          AND block_time BETWEEN TIMESTAMP '2025-10-08' AND TIMESTAMP '2025-10-11'
        ORDER BY block_time ASC
    """
    },
    "iran": {
    "label": "CONFIRMED",
    "sql": """
        SELECT block_time, question, asset_id, price, amount, shares, maker, taker, token_outcome_name
        FROM polymarket_polygon.market_trades
        WHERE question = 'US strikes Iran by February 28, 2026?'
          AND block_time BETWEEN TIMESTAMP '2026-02-26' AND TIMESTAMP '2026-02-28'
        ORDER BY block_time ASC
    """
},

}

dune_results = {}

for name, config in QUERIES.items():
    print(f"\n── {name.upper()} ({config['label']}) ──────────────────")
    try:
        exec_id = execute_sql(config["sql"])
        print(f"  Execution ID: {exec_id}")
        success = poll_until_done(exec_id)
        if success:
            df = fetch_results(exec_id)
            dune_results[name] = df
            print(f"  ✅ Got {len(df)} rows")
            if len(df) > 0:
                print(f"  Time range: {df['block_time'].min()} → {df['block_time'].max()}")
                print(f"  Unique wallets: {pd.concat([df['maker'], df['taker']]).nunique()}")
        else:
            dune_results[name] = pd.DataFrame()
    except Exception as e:
        print(f"  ❌ Error: {e}")
        dune_results[name] = pd.DataFrame()

# ── Step 3: Summary ──────────────────────────────────────────────────────────
print("\n── Summary ──────────────────────────────────────────────────────")
for name, df in dune_results.items():
    print(f"  {name}: {len(df)} rows")
print("\nNote: Nobel and Israel/Iran markets pending exact question text from Step 1 output")


# Find Iran bombing market question text
iran_sql = """
    SELECT DISTINCT question, COUNT(*) as trade_count
    FROM polymarket_polygon.market_trades
    WHERE (
        question LIKE '%US strike Iran%'
        OR question LIKE '%strike Iran by Feb%'
        OR question LIKE '%Iran by February 28%'
    )
    AND block_time BETWEEN TIMESTAMP '2026-02-26' AND TIMESTAMP '2026-02-28'
    GROUP BY question
    ORDER BY trade_count DESC
    LIMIT 10
"""
exec_id = execute_sql(iran_sql)
poll_until_done(exec_id)
df_iran = fetch_results(exec_id)
print(df_iran[["question", "trade_count"]].to_string())

── Step 1: Finding exact question text ──────────────────────────
  Status: QUERY_STATE_PENDING — waiting 5s...
  Status: QUERY_STATE_EXECUTING — waiting 5s...
                                                         question  trade_count
0                        Israel strikes Iran by January 31, 2026?       138944
1              Will US or Israel strike Iran by January 31, 2026?        48037
2                            Will US or Israel strike Iran first?        41978
3                          Israel strikes Iran by March 31, 2026?        41736
4                       Will Axiom be accused of insider trading?        38497
5             Will US or Israel strike Iran by February 28, 2026?        36183
6                       Israel strikes Iran by February 28, 2026?        31465
7        Will María Corina Machado enter Venezuela by January 31?        30404
8             Will Trump and Machado share the Nobel Peace Prize?        22184
9             Will US or Israel strike Iran by Feb

In [93]:
#@title 12 - Compute wallet-level features for labeled markets

def compute_wallet_features(df, resolution_time_str):
    """
    Compute wallet-level features from raw trade data.

    Args:
        df: DataFrame with columns block_time, maker, taker, amount, price, token_outcome_name
        resolution_time_str: ISO string of market resolution time
    """
    if len(df) == 0:
        return None

    # Parse times
    df = df.copy()
    df["block_time"] = pd.to_datetime(df["block_time"], utc=True)
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0)
    df["price"] = pd.to_numeric(df["price"], errors="coerce").fillna(0)

    resolution_time = pd.Timestamp(resolution_time_str, tz="UTC")
    final_12h_start = resolution_time - pd.Timedelta(hours=12)

    # Use maker as the primary wallet identifier
    df["wallet"] = df["maker"]
    total_volume = df["amount"].sum()

    # ── Feature 1: Wallet concentration ─────────────────────────────────────
    wallet_volume = df.groupby("wallet")["amount"].sum().sort_values(ascending=False)
    top3_volume = wallet_volume.head(3).sum()
    wallet_concentration = top3_volume / total_volume if total_volume > 0 else 0

    # ── Feature 2: New wallet ratio ──────────────────────────────────────────
    # "New" = wallet whose first trade in our window is in the final 12h
    wallet_first_trade = df.groupby("wallet")["block_time"].min()
    new_wallets = wallet_first_trade[wallet_first_trade >= final_12h_start].index
    new_wallet_volume = df[df["wallet"].isin(new_wallets)]["amount"].sum()
    new_wallet_ratio = new_wallet_volume / total_volume if total_volume > 0 else 0

    # ── Feature 2b: New wallet ratio (final 6h) ──────────────────────────────
    final_6h_start = resolution_time - pd.Timedelta(hours=6)
    new_wallets_6h = wallet_first_trade[wallet_first_trade >= final_6h_start].index
    new_wallet_volume_6h = df[df["wallet"].isin(new_wallets_6h)]["amount"].sum()
    new_wallet_ratio_6h = new_wallet_volume_6h / total_volume if total_volume > 0 else 0

    # ── Feature 3: Burst trading score ──────────────────────────────────────
    # Max trades any single wallet placed in a 1-hour window
    df["hour_bucket"] = df["block_time"].dt.floor("h")
    trades_per_wallet_hour = df.groupby(["wallet", "hour_bucket"]).size()
    burst_score = trades_per_wallet_hour.max() if len(trades_per_wallet_hour) > 0 else 0

    # ── Feature 4: Directional consensus ────────────────────────────────────
    # % of volume on the winning outcome
    outcome_volumes = df.groupby("token_outcome_name")["amount"].sum()
    if len(outcome_volumes) > 0:
        winning_outcome_volume = outcome_volumes.max()
        directional_consensus = winning_outcome_volume / total_volume if total_volume > 0 else 0
    else:
        directional_consensus = 0

    # ── Feature 5: Trade-direction VPIN ─────────────────────────────────────
    # Use price relative to 0.5 to classify buy/sell pressure
    # Trades above 0.5 = buying YES (bullish), below 0.5 = buying NO (bearish)
    df_sorted = df.sort_values("block_time")
    prices = df_sorted["price"].values
    amounts = df_sorted["amount"].values

    buy_volume = amounts[prices > 0.5].sum()
    sell_volume = amounts[prices <= 0.5].sum()
    total_directional = buy_volume + sell_volume

    trade_vpin = abs(buy_volume - sell_volume) / total_directional if total_directional > 0 else 0

    return {
        "total_volume": total_volume,
        "unique_wallets": df["wallet"].nunique(),
        "wallet_concentration": wallet_concentration,
        "new_wallet_ratio": new_wallet_ratio,
        "burst_score": int(burst_score),
        "directional_consensus": directional_consensus,
        "trade_vpin": trade_vpin,
        "top3_wallets": list(wallet_volume.head(3).index),
        "top3_volume_usd": round(top3_volume, 2),
        "new_wallet_ratio_6h": new_wallet_ratio_6h
    }


# Resolution times for each market
RESOLUTION_TIMES = {
    "maduro":  "2026-01-31T00:00:00",
    "zachxbt": "2026-02-26T23:59:00",
    "nobel":   "2025-10-10T11:00:00",
    "iran": "2026-02-28T23:59:00"

}

# Compute features for each market
print("── Wallet-level features ────────────────────────────────────────\n")
wallet_features = {}

for name, df in dune_results.items():
    if len(df) == 0:
        print(f"⚠️  {name}: no data\n")
        continue

    label = {"maduro": "SUSPECTED", "zachxbt": "SUSPECTED", "nobel": "CONFIRMED", "iran": "CONFIRMED"}[name]
    features = compute_wallet_features(df, RESOLUTION_TIMES[name])

    if features is None:
        continue

    wallet_features[name] = features

    print(f"── {name.upper()} ({label}) ──────────────────────────────")
    print(f"  Total volume:          ${features['total_volume']:,.2f}")
    print(f"  Unique wallets:        {features['unique_wallets']:,}")
    print(f"  Wallet concentration:  {features['wallet_concentration']:.1%}  (top 3 = {features['wallet_concentration']*100:.1f}% of volume)")
    print(f"  New wallet ratio:      {features['new_wallet_ratio']:.1%}  (% volume from wallets first seen in final 12h)")
    print(f"  Burst score:           {features['burst_score']}  (max trades by one wallet in 1h)")
    print(f"  Directional consensus: {features['directional_consensus']:.1%}  (% volume on winning side)")
    print(f"  Trade VPIN:            {features['trade_vpin']:.3f}")
    print(f"  Top 3 wallets:         {features['top3_wallets'][0][:12]}... (${features['top3_volume_usd']:,.0f} combined)")
    print(f"  New wallet ratio (6h): {features['new_wallet_ratio_6h']:.1%}  (% volume from wallets first seen in final 6h)")
    print()

── Wallet-level features ────────────────────────────────────────

── MADURO (SUSPECTED) ──────────────────────────────
  Total volume:          $227,560.78
  Unique wallets:        2,132
  Wallet concentration:  30.8%  (top 3 = 30.8% of volume)
  New wallet ratio:      3.6%  (% volume from wallets first seen in final 12h)
  Burst score:           63  (max trades by one wallet in 1h)
  Directional consensus: 33.6%  (% volume on winning side)
  Trade VPIN:            0.937
  Top 3 wallets:         0xed107a85a4... ($70,175 combined)
  New wallet ratio (6h): 0.1%  (% volume from wallets first seen in final 6h)

── ZACHXBT (SUSPECTED) ──────────────────────────────
  Total volume:          $10,162,243.14
  Unique wallets:        3,538
  Wallet concentration:  20.8%  (top 3 = 20.8% of volume)
  New wallet ratio:      10.4%  (% volume from wallets first seen in final 12h)
  Burst score:           270  (max trades by one wallet in 1h)
  Directional consensus: 50.3%  (% volume on winning side)

In [94]:
#@title 13 - Integrate wallet features into suspicion scoring

# ── 1. Define wallet suspicion thresholds ────────────────────────────────────
# Based on what we observed across our 4 labeled markets

def compute_wallet_suspicion_score(features):
    """
    Rule-based wallet suspicion score (0-1).
    Each component contributes equally (0.2 each).
    Thresholds derived from labeled market analysis.
    """
    score = 0.0

    # New wallet ratio 12h > 10% is elevated, > 50% is very suspicious
    nwr = features["new_wallet_ratio"]
    if nwr > 0.50:
        score += 0.20
    elif nwr > 0.10:
        score += 0.10

    # New wallet ratio 6h > 5% is notable
    nwr_6h = features["new_wallet_ratio_6h"]
    if nwr_6h > 0.10:
        score += 0.20
    elif nwr_6h > 0.05:
        score += 0.10

    # Trade VPIN > 0.7 is highly one-directional
    vpin = features["trade_vpin"]
    if vpin > 0.80:
        score += 0.20
    elif vpin > 0.60:
        score += 0.10

    # Burst score > 200 suggests algorithmic/bot behavior
    burst = features["burst_score"]
    if burst > 200:
        score += 0.20
    elif burst > 50:
        score += 0.10

    # Directional consensus > 70% means market was very one-sided
    dc = features["directional_consensus"]
    if dc > 0.70:
        score += 0.20
    elif dc > 0.55:
        score += 0.10

    return round(score, 2)


# ── 2. Score labeled markets ──────────────────────────────────────────────────
labels = {
    "maduro": "SUSPECTED",
    "zachxbt": "SUSPECTED",
    "nobel": "CONFIRMED",
    "iran": "CONFIRMED",
}

print("── Wallet suspicion scores (labeled markets) ────────────────────\n")
wallet_scores = []

for name, features in wallet_features.items():
    ws = compute_wallet_suspicion_score(features)
    wallet_scores.append({
        "market": name,
        "label": labels[name],
        "wallet_suspicion": ws,
        "new_wallet_12h": features["new_wallet_ratio"],
        "new_wallet_6h": features["new_wallet_ratio_6h"],
        "trade_vpin": features["trade_vpin"],
        "burst_score": features["burst_score"],
        "directional_consensus": features["directional_consensus"],
    })
    print(f"  {name.upper()} ({labels[name]}): wallet suspicion = {ws:.2f}")

df_wallet_scores = pd.DataFrame(wallet_scores)

# ── 3. Compare to price-based scores for overlapping markets ─────────────────
# Check if any of our labeled markets appear in df_scored
print("\n── Cross-checking against price-based model ─────────────────────\n")

market_name_map = {
    "maduro":  "Maduro",
    "zachxbt": "Axiom",
    "nobel":   "Machado",
    "iran":    "US strikes Iran",
}

for name in wallet_features:
    keyword = market_name_map[name]
    matches = df_scored[df_scored["question"].str.contains(keyword, case=False, na=False)]
    if len(matches) > 0:
        print(f"  {name.upper()} found in df_scored:")
        print(matches[["question", "suspicion_score"]].to_string())
    else:
        print(f"  {name.upper()} — not in df_scored (filtered out or no price history)")
    print()

# ── 4. Summary table ──────────────────────────────────────────────────────────
print("\n── Summary: wallet vs price-based scores ────────────────────────\n")
print(f"{'Market':<12} {'Label':<12} {'Wallet Score':<15} {'Price Score':<15} {'Agreement'}")
print("─" * 65)

for _, row in df_wallet_scores.iterrows():
    name = row["market"]
    keyword = market_name_map[name]
    matches = df_scored[df_scored["question"].str.contains(keyword, case=False, na=False)]
    price_score = f"{matches['suspicion_score'].max():.3f}" if len(matches) > 0 else "N/A"

    if price_score != "N/A":
        agreement = "✅ Both flag" if row["wallet_suspicion"] > 0.2 and float(price_score) > 0.02 else \
                    "⚠️  Wallet only" if row["wallet_suspicion"] > 0.2 else \
                    "⚠️  Price only" if float(price_score) > 0.02 else \
                    "❌ Neither flags"
    else:
        agreement = "⚠️  Wallet only (no price data)" if row["wallet_suspicion"] > 0.2 else "❌ Neither flags"

    print(f"{name:<12} {row['label']:<12} {row['wallet_suspicion']:<15.2f} {price_score:<15} {agreement}")

print("""
── Key thresholds used ───────────────────────────────────────────
  New wallet ratio 12h: >50% = high, >10% = elevated
  New wallet ratio 6h:  >10% = high, >5% = elevated
  Trade VPIN:           >0.80 = high, >0.60 = elevated
  Burst score:          >200 = high, >50 = elevated
  Directional consensus:>70% = high, >55% = elevated
""")

── Wallet suspicion scores (labeled markets) ────────────────────

  MADURO (SUSPECTED): wallet suspicion = 0.30
  ZACHXBT (SUSPECTED): wallet suspicion = 0.40
  NOBEL (CONFIRMED): wallet suspicion = 0.60
  IRAN (CONFIRMED): wallet suspicion = 0.40

── Cross-checking against price-based model ─────────────────────

  MADURO — not in df_scored (filtered out or no price history)

  ZACHXBT — not in df_scored (filtered out or no price history)

  NOBEL — not in df_scored (filtered out or no price history)

  IRAN — not in df_scored (filtered out or no price history)


── Summary: wallet vs price-based scores ────────────────────────

Market       Label        Wallet Score    Price Score     Agreement
─────────────────────────────────────────────────────────────────
maduro       SUSPECTED    0.30            N/A             ⚠️  Wallet only (no price data)
zachxbt      SUSPECTED    0.40            N/A             ⚠️  Wallet only (no price data)
nobel        CONFIRMED    0.60            N/A  